# 02 · Baseline Recommenders — The Engagement Trap

Four algorithms from the lab experiment list, plus a popularity reference line:

| Model | Lab experiment | Idea |
|---|---|---|
| **LinearRegression** | Exp 1 – Linear Regression | rating ≈ b₀ + b_user + b_movie, fitted with Ridge on one-hot user/movie columns |
| **ItemKNN** | Exp 8 – KNN | rating ≈ user mean + weighted average of the user's ratings on the *k* most similar movies (cosine on mean-centred ratings, shrunk when few users overlap) |
| **SVD** | Exp 6 – SVD | truncated SVD of the user-mean-centred rating matrix, R_c ≈ U Σ Vᵀ |
| **ContentPCA** | Exp 5 – PCA | movie features (genres + decade) → PCA; each user gets a ridge regression from those components to their ratings |
| Popularity *(reference)* | — | recommend the most-rated movies |

**Protocol.** 3 seeds × per-user 80/20 split. For each seed and model, hyperparameters are tuned on a validation split carved out of train (selection metric NDCG@10), the chosen setting is refit on all of train, and evaluated once on test. Results are mean ± std over the 3 seeds.

**Metrics** (top-10 lists never include movies the user already rated in train):
- *Rating accuracy:* RMSE, MAE on test ratings (predictions clipped to 0.5–5).
- *Ranking accuracy:* Precision@10, Recall@10, NDCG@10 — a test movie is *relevant* if rated ≥ 4.
- *Beyond accuracy:* Coverage (share of the catalogue that appears in any list), Gini of exposure, ILD (genre diversity inside a list), AvgPop (mean training-popularity of recommended movies).

In [ ]:
import os, sys, glob, json, time
# On Kaggle, etlib.py comes from an attached dataset; locally it sits next to this notebook.
for hit in glob.glob("/kaggle/input/**/etlib.py", recursive=True):
    sys.path.insert(0, os.path.dirname(hit))
    break
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import etlib as et

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
OUT = et.output_dir()
DATA_DIR = et.find_data_dir()
ratings, items, GENRES = et.load_movielens(DATA_DIR)  # movies with >= 5 ratings
G = et.genre_matrix(items, GENRES)
print("data:", DATA_DIR, "| outputs:", OUT)
print(len(ratings), "ratings |", ratings.user.nunique(), "users |", ratings.item.nunique(), "movies")


def save_fig(fig, name):
    fig.savefig(f"{OUT}/figures/{name}.png", dpi=150, bbox_inches="tight")

## 1. Configuration

In [ ]:
SEEDS = [0, 1, 2]
K = 10
TUNE_METRIC = "NDCG@10"
F = et.item_features(items, GENRES)

# name: (factory(param, seed) -> unfitted model, hyperparameter grid)
MODELS = {
    "LinearRegression": (lambda p, seed: et.BiasRidge(alpha=p), [10, 25, 50, 100, 250, 500]),
    "ItemKNN": (lambda p, seed: et.ItemKNN(k=p), [5, 10, 20, 40, 80]),
    "SVD": (lambda p, seed: et.SVDRec(k=p, seed=seed), [5, 10, 20, 40, 60]),
    "ContentPCA": (lambda p, seed: et.ContentPCA(F, n_components=p[0], alpha=p[1]),
                   [(4, 1.0), (8, 1.0), (12, 1.0), (8, 10.0), (12, 10.0)]),
}

## 2. Tune on validation, evaluate on test

For each seed: `train/test` split → `fit/val` split inside train → grid search on `val` → refit best setting on `train` → evaluate on `test`. The test split is never used to choose anything.

In [ ]:
test_rows, tune_rows = [], []
start = time.time()
for seed in SEEDS:
    train, test = et.split_per_user(ratings, 0.2, seed)
    fit, val = et.split_per_user(train, 0.2, seed + 100)  # validation carved from train only
    R_fit, R_train = et.to_csr(fit), et.to_csr(train)

    pop = et.Popularity().fit(R_train)
    test_rows.append({"model": "Popularity (reference)", "seed": seed,
                      **et.evaluate(pop, train, test, G, K)})

    for name, (make, grid) in MODELS.items():
        seed_tune = []
        for p in grid:
            res = et.evaluate(make(p, seed).fit(R_fit), fit, val, G, K)
            seed_tune.append({"model": name, "seed": seed, "param": str(p), **res})
        tune_rows += seed_tune
        best = int(np.argmax([r[TUNE_METRIC] for r in seed_tune]))
        model = make(grid[best], seed).fit(R_train)
        test_rows.append({"model": name, "seed": seed, "param": str(grid[best]),
                          "val_" + TUNE_METRIC: seed_tune[best][TUNE_METRIC],
                          **et.evaluate(model, train, test, G, K)})
        print(f"seed {seed} | {name:16s} | best param {grid[best]!s:10s} | {time.time() - start:5.0f}s")

### Validation results (mean over seeds) and chosen hyperparameters

In [ ]:
tune = pd.DataFrame(tune_rows)
tune.to_csv(f"{OUT}/results/02_tuning_validation.csv", index=False)
display(tune.groupby(["model", "param"], sort=False)[["RMSE", "Precision@10", "NDCG@10", "Coverage"]]
        .mean().round(4))
results = pd.DataFrame(test_rows)
results.pivot(index="seed", columns="model", values="param").drop(columns="Popularity (reference)")

## 3. Test results (mean ± std over 3 seeds)

In [ ]:
results.to_csv(f"{OUT}/results/02_baselines_per_seed.csv", index=False)
table, agg = et.mean_std_table(test_rows)
table = table.drop(columns="val_" + TUNE_METRIC)
table.to_csv(f"{OUT}/results/02_baselines_mean_std.csv")
table

In [ ]:
METRICS = ["RMSE", "MAE", "Precision@10", "Recall@10", "NDCG@10", "Coverage", "Gini", "ILD", "AvgPop"]
names = agg.index.tolist()
short = [n.replace(" (reference)", "*") for n in names]
colors = ["gray"] + [f"C{i}" for i in range(len(names) - 1)]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, m in zip(axes.ravel(), METRICS):
    ax.bar(range(len(names)), agg[m]["mean"], yerr=agg[m]["std"], capsize=4, color=colors)
    ax.set_xticks(range(len(names)), short, rotation=25, ha="right")
    ax.set_title(m + ("  (lower is better)" if m in ("RMSE", "MAE", "Gini", "AvgPop") else ""))
fig.suptitle("Baselines on test (mean ± std, 3 seeds). * = reference line")
fig.tight_layout()
save_fig(fig, "02_baselines_bars")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, x in zip(axes, ["AvgPop", "Coverage"]):
    for j, n in enumerate(names):
        ax.errorbar(agg[x]["mean"][n], agg["NDCG@10"]["mean"][n],
                    xerr=agg[x]["std"][n], yerr=agg["NDCG@10"]["std"][n], fmt="o", color=colors[j])
        ax.annotate(short[j], (agg[x]["mean"][n], agg["NDCG@10"]["mean"][n]),
                    textcoords="offset points", xytext=(6, 4))
    ax.set(xlabel=x, ylabel="NDCG@10")
axes[0].set_title("Accuracy vs popularity of what is recommended")
axes[1].set_title("Accuracy vs catalogue coverage")
fig.tight_layout()
save_fig(fig, "02_accuracy_vs_popularity")
plt.show()

## 4. Best model

The best model is chosen by **mean validation NDCG@10** (the same metric used for tuning), so this choice never looks at test. Its test scores are reported above. The model with the lowest test RMSE is printed too, because rating accuracy and ranking quality do not always agree.

If the validation winner is *not* the test winner, report both and keep the rule. Changing the rule after seeing test results would be tuning on test.

Notebooks 03 and 04 build on **SVD** for a design reason, whichever model wins here. The re-rankers need *personalised* scores and *movie embeddings*. LinearRegression gives every user almost the same list and has no embeddings, so there would be nothing to re-rank or cluster.

In [ ]:
ranked = results.dropna(subset=["val_" + TUNE_METRIC]).groupby("model")["val_" + TUNE_METRIC].mean()
best_model = ranked.idxmax()
print("Mean validation NDCG@10:\n", ranked.sort_values(ascending=False).round(4).to_string())
print("\nBest model (validation NDCG@10):", best_model)
print("Lowest test RMSE              :", agg["RMSE"]["mean"].drop("Popularity (reference)").idxmin())

**Things to look at when you write up the results**
- Does the lowest-RMSE model also give the best top-10 lists? If not, that is the project's theme in miniature: the number a model is trained on is not the outcome we care about.
- How does each model's AvgPop and Coverage compare with the popularity reference? A model can score well on NDCG partly *because* it recommends popular movies (they are more likely to be in anyone's test set).
- LinearRegression gives every user the **same** ranking (b_user only shifts all scores equally), so its lists differ only where a user has already rated a top movie, and its Coverage stays tiny.
- The std over seeds tells you whether differences between models are bigger than split-to-split noise.

## 5. Export for the demo site

In [ ]:
export = {n: {m: {"mean": et.json_num(agg[m]["mean"][n]), "std": et.json_num(agg[m]["std"][n])}
              for m in METRICS} for n in names}
et.save_json({"seeds": SEEDS, "k": K, "best_model": best_model, "models": export},
             f"{OUT}/demo_data/baselines.json")
print("saved", f"{OUT}/demo_data/baselines.json", "| total time", round(time.time() - start), "s")